In [1]:
from datasets import load_dataset
from google.colab import drive
drive.mount('/drive')



Mounted at /drive


Generating train split: 0 examples [00:00, ? examples/s]

In [15]:
def read_conll_2003(file_path):
    sentences = []
    current_sentence = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            # Skip CoNLL document boundary markers
            if line.startswith("-DOCSTART-"):
                continue

            if not line:
                # Empty line signifies the end of a sentence
                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
            else:
                # Split columns: word, POS tag, chunk tag, NER tag
                splits = line.split()
                if len(splits) >= 4:
                    word = splits[0]        # Added [0]
                    pos_tag = splits[1]     # Added [1]
                    chunk_tag = splits[2]   # Added [2]
                    ner_tag = splits[3]     # Added [3]
                    current_sentence.append((word, pos_tag, chunk_tag, ner_tag))
        # Append the last sentence if the file doesn't end with an empty line
        if current_sentence:
            sentences.append(current_sentence)

    return sentences

# Load the file from your Colab workspace
file_path = '/drive/MyDrive/Start_LLM/eng.train'
dataset = read_conll_2003(file_path)

# Print out the first sentence to verify the structure
print(f"Total Sentences Loaded: {len(dataset)}")
print("First Sentence Sample:", dataset[0])


Total Sentences Loaded: 14041
First Sentence Sample: [('EU', 'NNP', 'B-NP', 'B-ORG'), ('rejects', 'VBZ', 'B-VP', 'O'), ('German', 'JJ', 'B-NP', 'B-MISC'), ('call', 'NN', 'I-NP', 'O'), ('to', 'TO', 'B-VP', 'O'), ('boycott', 'VB', 'I-VP', 'O'), ('British', 'JJ', 'B-NP', 'B-MISC'), ('lamb', 'NN', 'I-NP', 'O'), ('.', '.', 'O', 'O')]


In [16]:
import pandas as pd

# Flatten the parsed list for tabular representation
flat_data = []
for sentence_id, sentence in enumerate(dataset):
    for word, pos, chunk, ner in sentence:
        flat_data.append([sentence_id, word, pos, chunk, ner])

df = pd.DataFrame(flat_data, columns=['Sentence_ID', 'Word', 'POS', 'Chunk', 'NER_Tag'])

# Group the dataframe by Sentence_ID to reconstruct full text sentences
grouped = df.groupby("Sentence_ID").agg({
    "Word": list,
    "NER_Tag": list
}).reset_index()

# Create a human-readable text column just to look at it
grouped["Full_Sentence"] = grouped["Word"].apply(lambda tokens: " ".join(tokens))

Dataset = grouped[["Sentence_ID", "Full_Sentence", "Word", "NER_Tag"]]



In [17]:
type(Dataset)

pandas.core.frame.DataFrame

In [18]:
Dataset.head(10)

,Sentence_ID,Full_Sentence,Word,NER_Tag
0,0,EU rejects German call to boycott British lamb .,"[EU, rejects, German, call, to, boycott, Briti...","[B-ORG, O, B-MISC, O, O, O, B-MISC, O, O]"
1,1,Peter Blackburn,"[Peter, Blackburn]","[B-PER, I-PER]"
2,2,BRUSSELS 1996-08-22,"[BRUSSELS, 1996-08-22]","[B-LOC, O]"
3,3,The European Commission said on Thursday it di...,"[The, European, Commission, said, on, Thursday...","[O, B-ORG, I-ORG, O, O, O, O, O, O, B-MISC, O,..."
4,4,Germany 's representative to the European Unio...,"[Germany, 's, representative, to, the, Europea...","[B-LOC, O, O, O, O, B-ORG, I-ORG, O, O, O, B-P..."
5,5,""" We do n't support any such recommendation be...","["", We, do, n't, support, any, such, recommend...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
6,6,He said further scientific study was required ...,"[He, said, further, scientific, study, was, re...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
7,7,He said a proposal last month by EU Farm Commi...,"[He, said, a, proposal, last, month, by, EU, F...","[O, O, O, O, O, O, O, B-ORG, O, O, B-PER, I-PE..."
8,8,Fischler proposed EU-wide measures after repor...,"[Fischler, proposed, EU-wide, measures, after,...","[B-PER, O, B-MISC, O, O, O, O, B-LOC, O, B-LOC..."
9,9,But Fischler agreed to review his proposal aft...,"[But, Fischler, agreed, to, review, his, propo...","[O, B-PER, O, O, O, O, O, O, O, B-ORG, O, O, O..."


In [23]:
from datasets import Dataset as HfDataset

# Use the 'grouped' DataFrame which is confirmed to be a Pandas DataFrame
# and from which 'Dataset' was derived. This avoids potential name conflicts
# with the 'datasets.Dataset' class.
train_dataset = HfDataset.from_pandas(grouped)
train_dataset

Dataset({
    features: ['Sentence_ID', 'Word', 'NER_Tag', 'Full_Sentence'],
    num_rows: 14041
})

In [25]:
#train_dataset = Coll_Dataset['train']
from sklearn.model_selection import train_test_split

dataset_split = train_dataset.train_test_split(test_size=0.2, seed=42)
dataset_split["validation"] = dataset_split["test"]
del dataset_split["test"]

In [26]:
import collections
import pandas as pd


all_ner_tags = [tag for sublist in train_dataset['NER_Tag'] for tag in sublist]


ner_tag_counts = collections.Counter(all_ner_tags)


ner_tag_counts_df = pd.DataFrame(ner_tag_counts.items(), columns=['NER_Tag', 'Count']).sort_values(by='Count', ascending=False)

label_list = []
label_list = ner_tag_counts_df["NER_Tag"]

In [29]:
train_dataset[0]

{'Sentence_ID': 0,
 'Word': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'NER_Tag': ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O'],
 'Full_Sentence': 'EU rejects German call to boycott British lamb .'}

In [27]:
print(label_list)


1         O
5     B-LOC
3     B-PER
0     B-ORG
4     I-PER
6     I-ORG
2    B-MISC
8     I-LOC
7    I-MISC
Name: NER_Tag, dtype: object


In [37]:
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")



label_to_id = {label: i for i, label in enumerate(label_list)}
id_to_label = {i: label for i, label in enumerate(label_list)}

def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(
        examples["Word"],
        truncation=True,
        batched=False,
        padding="max_length",
        max_length=128,
        is_split_into_words=True
    )

    labels = []

    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None
    label_ids = []

    for word_idx in word_ids:
        if word_idx is None:

            label_ids.append(-100)
        elif word_idx != previous_word_idx:

            text_label = examples["NER_Tag"][word_idx]
            label_ids.append(label_to_id[text_label])
        else:
           label_ids.append(-100)
        previous_word_idx = word_idx

    tokenized_inputs["labels"] = label_ids
    return tokenized_inputs





In [38]:
output = tokenize_and_align_labels(train_dataset[0])

print("original words", train_dataset["Word"])
print(" tokenized:", output["input_ids"])
print("final tags", output["labels"])

original words Column([['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], ['Peter', 'Blackburn'], ['BRUSSELS', '1996-08-22'], ['The', 'European', 'Commission', 'said', 'on', 'Thursday', 'it', 'disagreed', 'with', 'German', 'advice', 'to', 'consumers', 'to', 'shun', 'British', 'lamb', 'until', 'scientists', 'determine', 'whether', 'mad', 'cow', 'disease', 'can', 'be', 'transmitted', 'to', 'sheep', '.'], ['Germany', "'s", 'representative', 'to', 'the', 'European', 'Union', "'s", 'veterinary', 'committee', 'Werner', 'Zwingmann', 'said', 'on', 'Wednesday', 'consumers', 'should', 'buy', 'sheepmeat', 'from', 'countries', 'other', 'than', 'Britain', 'until', 'the', 'scientific', 'advice', 'was', 'clearer', '.']])
 tokenized: [101, 7270, 22961, 1528, 1840, 1106, 21423, 1418, 2495, 12913, 119, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [46]:
tokenized_dataset = dataset_split.map(
    tokenize_and_align_labels, batched=False
)

tokenized_dataset

Map:   0%|          | 0/11232 [00:00<?, ? examples/s]

Map:   0%|          | 0/2809 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Sentence_ID', 'Word', 'NER_Tag', 'Full_Sentence', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 11232
    })
    validation: Dataset({
        features: ['Sentence_ID', 'Word', 'NER_Tag', 'Full_Sentence', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2809
    })
})

In [47]:
tokenized_dataset['train'].to_pandas().head(10)

,Sentence_ID,Word,NER_Tag,Full_Sentence,input_ids,token_type_ids,attention_mask,labels
0,2341,"[6., Frank, Asselman, (, Belgium, ), 13.64]","[O, B-PER, I-PER, O, B-LOC, O, O]",6. Frank Asselman ( Belgium ) 13.64,"[101, 127, 119, 2748, 1249, 11510, 1399, 113, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, ...","[-100, 0, -100, 2, 4, -100, -100, 0, 1, 0, 0, ..."
1,11787,"[Pay, Oct, 20]","[O, O, O]",Pay Oct 20,"[101, 22531, 14125, 1406, 102, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-100, 0, 0, 0, -100, -100, -100, -100, -100, ..."
2,9176,"[Beijing, has, called, on, Seoul, to, stop, So...","[B-LOC, O, O, O, B-LOC, O, O, B-MISC, I-MISC, ...",Beijing has called on Seoul to stop South Kore...,"[101, 6671, 1144, 1270, 1113, 10853, 1106, 183...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, 1, 0, 0, 0, 1, 0, 0, 6, 8, 0, 0, 0, 0, ..."
3,3512,"[+11, Steve, Schneiter, 77, 74, ), through, 18...","[O, B-PER, I-PER, O, O, O, O, O, O]",+11 Steve Schneiter 77 74 ) through 18 ),"[101, 116, 1429, 3036, 20452, 7272, 27863, 120...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, 0, -100, 2, 4, -100, -100, -100, 0, 0, ..."
4,6781,"[SOCCER, -, PLAYERS, LEAVE, MATCH, EARLY, TO, ...","[O, O, O, O, O, O, O, O, O, O]",SOCCER - PLAYERS LEAVE MATCH EARLY TO CATCH PL...,"[101, 156, 9244, 10954, 2069, 118, 153, 10783,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, 0, -100, -100, -100, 0, 0, -100, -100, ..."
5,6282,"[Dow, rises, on, Philip, Morris, ,, other, sto...","[B-MISC, O, O, B-ORG, I-ORG, O, O, O, O, O]","Dow rises on Philip Morris , other stocks lower .","[101, 26535, 9440, 1113, 4367, 5744, 117, 1168...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, ...","[-100, 6, 0, 0, 3, 5, 0, 0, 0, 0, 0, -100, -10..."
6,6919,"[NEW, YORK, 1996-08-26]","[B-LOC, I-LOC, O]",NEW YORK 1996-08-26,"[101, 26546, 2924, 162, 9565, 2428, 1820, 118,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, ...","[-100, 1, -100, 7, -100, -100, 0, -100, -100, ..."
7,14012,"[Leading, scores, after]","[O, O, O]",Leading scores after,"[101, 16425, 7432, 1170, 102, 0, 0, 0, 0, 0, 0...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-100, 0, 0, 0, -100, -100, -100, -100, -100, ..."
8,7214,"[Bowling, :, Streak, 10-1-50-1, (, 2w, ,, 2nb,...","[O, O, B-PER, O, O, O, O, O, O, O, B-PER, O, O...","Bowling : Streak 10-1-50-1 ( 2w , 2nb ) , Bran...","[101, 19356, 131, 1457, 22362, 1275, 118, 122,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, 0, 0, 2, -100, 0, -100, -100, -100, -10..."
9,167,"[Hilary, Gush]","[B-PER, I-PER]",Hilary Gush,"[101, 25296, 13067, 1324, 102, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-100, 2, 4, -100, -100, -100, -100, -100, -10..."


In [52]:
sample = tokenized_dataset["train"][0]
len(sample["input_ids"])


128

In [53]:
len(sample["labels"])

128

In [55]:
print(sample["Word"])
print(sample["NER_Tag"])
print(sample["input_ids"])
print(sample["labels"])

['6.', 'Frank', 'Asselman', '(', 'Belgium', ')', '13.64']
['O', 'B-PER', 'I-PER', 'O', 'B-LOC', 'O', 'O']
[101, 127, 119, 2748, 1249, 11510, 1399, 113, 4990, 114, 1492, 119, 3324, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[-100, 0, -100, 2, 4, -100, -100, 0, 1, 0, 0, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -